In [1]:
# =============================================================================
# QFID · Q1 + SUPERVISED CheXpert PRE-ADAPTATION + INFORMED ROUTING -> QUANTUM FEW-SHOT
# Cell 1/4 — config, environment, cohorts
# -----------------------------------------------------------------------------
# Pipeline
#   Cell 1  config, helpers, cohorts
#   Cell 2  supervised pre-adaptation on the 1.6k labelled frontal CheXpert cohort
#           (the SAME patients as Q1's 'train' cohort) of all three pretrained encoders:
#             T0_NIH_whole_vit.pt  (whole image)   T0_NIH_patch_vit.pt (six crops, MIL)
#             CheXzero             (zero-shot text-anchor prior)
#           then feature caches for original AND pre-adapted encoders
#   Cell 3  informed class-wise routing whole <-> patch from the checkpoints' own
#           per-class NIH validation AUROC (per_class_validation_metrics.csv),
#           routed (N,5,128) features, frozen audit
#   Cell 4  Q1 heads, unchanged (classical matched / quantum VQC / hybrid), K-shot
#           fine-tuning WITH LABELS, CheXzero zero-shot residual prior, paired tests
# Leakage rule: the 1.6k pre-adaptation patients are never used as few-shot support.
#   Supports come from a separate, patient-disjoint 'support' cohort (same size, same
#   balanced sampler). select/dev/final/expert_valid are disjoint from both (asserted).
# Reporting: results are "K-shot after supervised target pre-adaptation".
# Run RUN_MODE="SMOKE" first (~10 min: tiny cohorts, 2 pre-adaptation steps per branch,
# 4-qubit/1-layer circuit, 2 epochs, 1-shot x 2 episodes) -> initial result tables that only
# prove the pipeline runs end to end. Then set RUN_MODE="FULL" and Run All.
# Results need supervisor review.
# =============================================================================
from __future__ import annotations
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import sys, json, math, time, random, hashlib, re, subprocess, copy, warnings
from pathlib import Path
from itertools import product
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from torchvision.transforms import functional as TF
from torchvision.models import vit_b_16
from sklearn.metrics import roc_auc_score, average_precision_score
from scipy.stats import t as student_t

try:
    import open_clip
except ImportError:   # needs Kaggle internet ON
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "open_clip_torch"], check=True)
    import open_clip
from open_clip.model import build_model_from_openai_state_dict

# ----------------------------------------------------------------------------
# 1. Paths
# ----------------------------------------------------------------------------
STAGE1_ROOT   = Path("/kaggle/input/datasets/zarinn/nih-checkpoint-vit/qfid_matched_stage1_nih_only")
T0_WHOLE_PATH = STAGE1_ROOT / "T0_NIH_whole_vit.pt"
T0_PATCH_PATH = STAGE1_ROOT / "T0_NIH_patch_vit.pt"
ROUTING_CSV   = STAGE1_ROOT / "per_class_validation_metrics.csv"
CHEXZERO_CKPT = Path("/kaggle/input/models/anikazarin/chexzero/pytorch/default/1/best_64_5e-05_original_22000_0.864.pt")
CHEXPERT_ROOT = Path("/kaggle/input/datasets/ashery/chexpert")
OUT = Path("/kaggle/working/qfid_preadapt_routed"); OUT.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------------------------
# 2. Settings
# ----------------------------------------------------------------------------
RUN_MODE = "FULL"                      # "SMOKE" -> "FULL"
SMOKE    = RUN_MODE == "SMOKE"
SEED     = 42
CLASSES    = ("Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Pleural Effusion")
LABEL_COLS = tuple(f"label_{i}" for i in range(len(CLASSES)))
LABEL_POLICY = "mask"                   # Q1 policy: CheXpert -1 and blank ignored

# Cohorts (patients). 'preadapt' == Q1's 'train' cohort (same function, seed, cap).
CAP = dict(preadapt=40, support=40, select=40, dev=40, final=40) if SMOKE else \
      dict(preadapt=1600, support=1600, select=400, dev=400, final=400)
SPLIT_FRACTIONS  = {"train": .55, "select": .15, "dev": .15, "final": .15}
USE_EXPERT_VALID = not SMOKE        # SMOKE skips the 234-image expert set to stay under ~10 min

# Supervised pre-adaptation (labels of the 1.6k cohort)
ADAPT_EPOCHS   = 1 if SMOKE else 5
ADAPT_BLOCKS   = 2                      # last N transformer blocks + final LN + head are trainable
LR_BACKBONE, LR_HEAD = 1e-5, 1e-4
LAMBDA_ANCHOR  = 0.1                    # 1 - cos(student emb, original emb): limits forgetting
BATCH_WHOLE, BATCH_PATCH, BATCH_CXZ = (8, 4, 8) if SMOKE else (32, 8, 64)
SMOKE_MAX_STEPS = 2                     # SMOKE: pre-adaptation steps per branch
SMOKE_BUDGET_MIN = 10
ENCODER_SETS   = ("original", "preadapted")   # both are carried through Cell 4 (paired)

# Routing
ROUTING_SOURCE   = "nih_val_csv"        # "nih_val_csv" (checkpoint validation AUROC) | "select" (Q1 rule on CheXpert SELECT) | "equal"
ROUTING_DEADBAND = 0.03
PATCH_POOLING    = "attention"          # ONE crop-pooling rule for pre-adaptation, routing, audit and features
                                        # ("attention" = Q1 softmax(logit)*logit; "lme" = log-mean-exp MIL)
TARGET_CROP_BOXES = [(.05,.05,.55,.95), (.45,.05,.95,.95), (.10,.02,.90,.58), (.10,.42,.90,.98), (.18,.18,.82,.82), (.28,.08,.72,.92)]

# Runtime
EVAL_BATCH, WORKERS = 16, 0             # workers=0 avoids Kaggle loader deadlocks (Q1)
MAX_RUNTIME_HOURS = 11.0

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU (Settings -> Accelerator -> GPU T4), then Run All.")
DEVICE = torch.device("cuda:0"); torch.cuda.set_device(DEVICE)
def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(SEED)
RUN_T0 = time.time()
def elapsed(): return f"t+{(time.time()-RUN_T0)/60:.1f} min"
print("GPU:", torch.cuda.get_device_name(0), "| RUN_MODE:", RUN_MODE)
for p in (T0_WHOLE_PATH, T0_PATCH_PATH, ROUTING_CSV, CHEXZERO_CKPT):
    if not p.is_file(): raise FileNotFoundError(f"missing input: {p}")

def sha_of(obj) -> str:
    return hashlib.sha256(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()
def file_sha256(path, block=8 << 20) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while chunk := f.read(block): h.update(chunk)
    return h.hexdigest()

# ----------------------------------------------------------------------------
# 3. Metrics / losses
# ----------------------------------------------------------------------------
def macro_metrics(y, s):
    y, s = np.asarray(y, float), np.asarray(s, float); per = {}
    for c, name in enumerate(CLASSES):
        m = np.isfinite(y[:, c]); ok = m.sum() and len(np.unique(y[m, c])) == 2
        per[name] = (roc_auc_score(y[m, c], s[m, c]), average_precision_score(y[m, c], s[m, c])) if ok else (np.nan, np.nan)
    au = [a for a, _ in per.values() if np.isfinite(a)]; ap = [p for _, p in per.values() if np.isfinite(p)]
    return {"macro_auroc": float(np.mean(au)) if au else np.nan, "macro_auprc": float(np.mean(ap)) if ap else np.nan, "per_class": per}
def macro_auc(y, s): return macro_metrics(y, s)["macro_auroc"]

def masked_bce(logits, y):
    known = torch.isfinite(y)
    raw = F.binary_cross_entropy_with_logits(logits.float(), torch.nan_to_num(y, nan=0.0), reduction="none")
    return (raw * known).sum() / known.sum().clamp_min(1)

# ----------------------------------------------------------------------------
# 4. Cohort helpers (verbatim Q1 logic)
# ----------------------------------------------------------------------------
def patient_from_path(p):
    m = re.search(r"(patient\d+)", str(p), re.I)
    return m.group(1).lower() if m else Path(str(p)).parts[-3].lower()

def partition_patients(df, fractions, seed):
    rng = np.random.default_rng(seed)
    Y = df[list(LABEL_COLS)].to_numpy(float)
    chars = np.where(~np.isfinite(Y), "u", np.where(Y >= .5, "1", "0"))
    patterns = pd.Series(["".join(row) for row in chars], index=df.index)
    groups = {k: v.to_numpy() for k, v in patterns.groupby(patterns).groups.items()}
    out = {k: [] for k in fractions}; target = {k: fractions[k] * len(df) for k in fractions}
    for _, idx in sorted(groups.items()):
        for i in rng.permutation(idx):
            key = min(fractions, key=lambda k: (len(out[k]) + 1) / (target[k] + 1e-9)); out[key].append(i)
    frames = {k: df.loc[v].reset_index(drop=True) for k, v in out.items()}
    ids = {k: set(v.patient_id.astype(str)) for k, v in frames.items()}
    for a in frames:
        for b in frames:
            if a < b and ids[a] & ids[b]: raise RuntimeError(f"patient leakage between {a} and {b}")
    return frames

def balanced_cap(df, cap, seed):
    if cap <= 0 or len(df) <= cap: return df.reset_index(drop=True)
    rng = np.random.default_rng(seed); chosen = []
    Y = df[list(LABEL_COLS)].to_numpy(float); per = max(2, cap // (2 * len(CLASSES)))
    for c in range(len(CLASSES)):
        for mask in (np.isfinite(Y[:, c]) & (Y[:, c] >= .5), np.isfinite(Y[:, c]) & (Y[:, c] < .5)):
            idx = np.flatnonzero(mask)
            if len(idx): chosen += rng.choice(idx, min(per, len(idx)), replace=False).tolist()
    chosen = list(dict.fromkeys(chosen))
    rest = np.setdiff1d(np.arange(len(df)), np.asarray(chosen, dtype=int))
    if len(chosen) < cap: chosen += rng.choice(rest, min(cap - len(chosen), len(rest)), replace=False).tolist()
    return df.iloc[sorted(chosen[:cap])].reset_index(drop=True)

def split_seed(name): return SEED + int(hashlib.sha256(name.encode()).hexdigest()[:6], 16) % 1000

def find_chexpert_csv(name):
    """Direct locations first; the recursive search (slow on Kaggle's network mount) only as a fallback."""
    for c in (CHEXPERT_ROOT / name, CHEXPERT_ROOT / "CheXpert-v1.0-small" / name, CHEXPERT_ROOT / "CheXpert-v1.0" / name):
        if c.is_file(): return c
    hits = sorted(CHEXPERT_ROOT.rglob(name)) if CHEXPERT_ROOT.is_dir() else []
    return hits[0] if hits else None

def resolve_cxp(p):
    q = Path(str(p)); cands = [q, CHEXPERT_ROOT / q]
    if len(q.parts) > 1: cands.append(CHEXPERT_ROOT / Path(*q.parts[1:]))
    cands.append(CHEXPERT_ROOT / "CheXpert-v1.0-small" / q)
    for x in cands:
        if x.is_file(): return str(x)
    return None

def load_chexpert(csv_path):
    df = pd.read_csv(csv_path)
    if "Frontal/Lateral" in df: df = df[df["Frontal/Lateral"].astype(str).str.lower().eq("frontal")].copy()   # frontal only
    labels = df[list(CLASSES)].apply(pd.to_numeric, errors="coerce").astype(float).mask(lambda x: x.eq(-1), np.nan)
    for j, col in enumerate(LABEL_COLS): df[col] = labels.iloc[:, j]
    df["patient_id"] = df["Path"].map(patient_from_path)
    df["_pick"] = [hashlib.sha256(f"{SEED}|{p}|{q}".encode()).hexdigest() for p, q in zip(df.patient_id, df.Path)]
    df = df.sort_values(["patient_id", "_pick"]).drop_duplicates("patient_id").drop(columns="_pick")
    return df[np.isfinite(df[list(LABEL_COLS)].to_numpy(float)).any(1)].reset_index(drop=True)

def attach_paths(df, name):
    df = df.copy(); df["resolved_path"] = [resolve_cxp(x) for x in df.Path]
    missing = int(df.resolved_path.isna().sum())
    if missing: raise FileNotFoundError(f"{name}: {missing} CheXpert images not found -- check CHEXPERT_ROOT.")
    return df.reset_index(drop=True)

# ----------------------------------------------------------------------------
# 5. Cohorts
# ----------------------------------------------------------------------------
train_csv = find_chexpert_csv("train.csv")
if train_csv is None: raise FileNotFoundError("CheXpert train.csv not found under CHEXPERT_ROOT.")
FULL_SPLIT = partition_patients(load_chexpert(train_csv), SPLIT_FRACTIONS, SEED)
COHORTS = {}
COHORTS["preadapt"] = attach_paths(balanced_cap(FULL_SPLIT["train"], CAP["preadapt"], split_seed("train")), "preadapt")  # == Q1 'train'
_pre_ids = set(COHORTS["preadapt"].patient_id.astype(str))
_rest = FULL_SPLIT["train"][~FULL_SPLIT["train"].patient_id.astype(str).isin(_pre_ids)].reset_index(drop=True)
COHORTS["support"] = attach_paths(balanced_cap(_rest, CAP["support"], split_seed("support")), "support")
for n in ("select", "dev"):
    COHORTS[n] = attach_paths(balanced_cap(FULL_SPLIT[n], CAP[n], split_seed(n)), n)
valid_csv = find_chexpert_csv("valid.csv") if USE_EXPERT_VALID else None
if valid_csv is not None: COHORTS["expert_valid"] = attach_paths(load_chexpert(valid_csv), "expert_valid")
elif USE_EXPERT_VALID: print("NOTE: valid.csv not found -- expert-label evaluation skipped.")

ids = {k: set(v.patient_id.astype(str)) for k, v in COHORTS.items()}
ids["final_locked"] = set(FULL_SPLIT["final"].patient_id.astype(str))
for a, b in product(ids, ids):
    if a < b and ids[a] & ids[b]: raise RuntimeError(f"patient overlap between {a} and {b}")
print("Cohorts (patients):", {k: len(v) for k, v in COHORTS.items()}, "| FINAL locked, never loaded")
YS = {n: COHORTS[n][list(LABEL_COLS)].to_numpy(float) for n in COHORTS}
cnt = [{"split": n, **{c: f"{int((YS[n][:, i] == 1).sum())}+/{int((YS[n][:, i] == 0).sum())}-" for i, c in enumerate(CLASSES)}} for n in COHORTS]
print("Known labels (pos/neg):\n" + pd.DataFrame(cnt).to_string(index=False))
EVAL_SPLITS = ["support", "select", "dev"] + (["expert_valid"] if "expert_valid" in COHORTS else [])
print("Cell 1 ready.", elapsed())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.5 MB/s eta 0:00:00
GPU: Tesla T4 | RUN_MODE: FULL
Cohorts (patients): {'preadapt': 1600, 'support': 1600, 'select': 400, 'dev': 400, 'expert_valid': 200} | FINAL locked, never loaded
Known labels (pos/neg):
       split Atelectasis Cardiomegaly Consolidation     Edema Pleural Effusion
    preadapt   336+/163-    313+/242-     203+/521- 369+/336-        475+/571-
     support    341+/71-    307+/247-     211+/464- 378+/351-        492+/552-
      select     79+/40-      80+/66-      48+/124-   90+/78-        114+/135-
         dev     87+/40-      71+/73-      54+/138-   94+/79-        127+/146-
expert_valid    75+/125-     66+/134-      32+/168-  42+/158-         64+/136-
Cell 1 ready. t+0.6 min


In [2]:
# =============================================================================
# Cell 2/4 — supervised pre-adaptation on the 1.6k labelled frontal CheXpert cohort
#   whole  : T0_NIH_whole_vit.pt, full image, masked BCE on its own path_head logits
#   patch  : T0_NIH_patch_vit.pt, the six fixed CheXpert crops, crop logits pooled per
#            image with pool_crops (PATCH_POOLING, identical to routing/audit in Cell 3)
#   CheXzero: logits = 100 * <z, t_pos - t_neg> (+bias), text anchors frozen, so the
#            adapted encoder still gives a valid zero-shot prior in Cell 4
#   Each: last ADAPT_BLOCKS blocks + final LN (+ head for T0) trainable, anchor to the
#   original embedding, best epoch by SELECT macro AUROC (original kept if none better).
#   Then: feature caches for BOTH original and pre-adapted encoders.
# =============================================================================
# ---- T0 ViT branches (Q1 definitions) ----
class DisentangleHead(nn.Module):
    def __init__(self, in_dim=768, shared=256, out=128, n_classes=5):
        super().__init__()
        self.shared   = nn.Sequential(nn.LayerNorm(in_dim), nn.Linear(in_dim, shared), nn.GELU(), nn.Dropout(.1), nn.LayerNorm(shared))
        self.causal   = nn.Sequential(nn.Linear(shared, out), nn.GELU(), nn.LayerNorm(out))
        self.spurious = nn.Sequential(nn.Linear(shared, out), nn.GELU(), nn.LayerNorm(out))
        self.path_head = nn.Linear(out, n_classes)
    def forward(self, h):
        s = self.shared(h); zc = F.normalize(self.causal(s), dim=-1)
        return {"zc": zc, "logits": self.path_head(zc)}

class ViTBranch(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = vit_b_16(weights=None); self.encoder.heads = nn.Identity(); self.head = DisentangleHead()
    def forward(self, x): return self.head(self.encoder(x))

def load_branch(path):
    payload = torch.load(path, map_location="cpu", weights_only=False)
    state = payload["model"] if isinstance(payload, dict) and "model" in payload else payload
    state = {k.removeprefix("module.").removeprefix("model."): v for k, v in state.items() if torch.is_tensor(v)}
    model = ViTBranch(); model.load_state_dict(state, strict=True)
    for p in model.parameters(): p.requires_grad_(False)
    return model.to(DEVICE).eval()

IMNET = ((.485, .456, .406), (.229, .224, .225))
EVAL_T  = T.Compose([T.Resize((224, 224)), T.ToTensor(), T.Normalize(*IMNET)])
TRAIN_T = T.Compose([T.RandomResizedCrop(224, scale=(.8, 1.), ratio=(.9, 1.1)), T.ToTensor(), T.Normalize(*IMNET)])
PRE_AUG = T.Compose([T.RandomRotation(5), T.ColorJitter(brightness=.2, contrast=.2)])   # no flips (laterality)

class T0DS(Dataset):
    """mode 'whole' -> (3,224,224); mode 'patch' -> (6,3,224,224) TARGET_CROP_BOXES. Labels may contain NaN."""
    def __init__(self, df, mode, train=False):
        self.paths, self.y, self.mode, self.train = df.resolved_path.tolist(), df[list(LABEL_COLS)].to_numpy(np.float32), mode, train
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        with Image.open(self.paths[i]) as z: im = z.convert("RGB")
        if self.train: im = PRE_AUG(im)
        if self.mode == "whole": x = (TRAIN_T if self.train else EVAL_T)(im)
        else:
            w, h = im.size
            x = torch.stack([EVAL_T(im.crop((int(a*w), int(b*h), int(c*w), int(d*h)))) for a, b, c, d in TARGET_CROP_BOXES])
        return x, torch.from_numpy(self.y[i])

def pool_crops(logits):   # (N,K,5) -> (N,5); the SAME rule is used in Cells 2, 3 and 4
    logits = logits.float()
    if PATCH_POOLING == "attention": return (torch.softmax(logits, dim=1) * logits).sum(1)
    if PATCH_POOLING == "lme":       return torch.logsumexp(logits, dim=1) - math.log(logits.shape[1])
    raise ValueError(PATCH_POOLING)

@torch.inference_mode()
def encode_t0(model, df, mode, tag=""):
    model.eval(); zc, lg = [], []
    for bi, (x, _) in enumerate(DataLoader(T0DS(df, mode), batch_size=EVAL_BATCH, shuffle=False, num_workers=WORKERS), 1):
        shape = x.shape[:2] if mode == "patch" else x.shape[:1]
        with torch.autocast("cuda", dtype=torch.float16):
            out = model(x.flatten(0, 1).to(DEVICE) if mode == "patch" else x.to(DEVICE))
        zc.append(out["zc"].float().view(*shape, -1).cpu()); lg.append(out["logits"].float().view(*shape, -1).cpu())
    return {"zc": torch.cat(zc), "logits": torch.cat(lg)}

def t0_scores(e): return pool_crops(e["logits"]) if e["logits"].dim() == 3 else e["logits"]   # one score per class per image

# ---- CheXzero (CLIP ViT-B/32) ----
def load_chexzero(path):
    sd = torch.load(path, map_location="cpu", weights_only=False)
    if isinstance(sd, dict) and isinstance(sd.get("state_dict"), dict): sd = sd["state_dict"]
    if isinstance(sd, dict) and isinstance(sd.get("model"), dict): sd = sd["model"]
    if not isinstance(sd, dict): sd = sd.state_dict()
    sd = {k.removeprefix("module."): v for k, v in sd.items() if torch.is_tensor(v)}
    model = build_model_from_openai_state_dict(sd).float().to(DEVICE).eval()
    for p in model.parameters(): p.requires_grad_(False)
    return model

# CheXzero preprocessing (as far as I know from its repo): grayscale, aspect-preserving pad to 320,
# 0-255 scale, normalise (101.48761, 83.43944), bicubic 224, 3 channels. SELECT zero-shot sanity below.
CXZ_MEAN, CXZ_STD = 101.48761, 83.43944
BICUBIC = T.InterpolationMode.BICUBIC
def pad_to_square(path, size=320):
    with Image.open(path) as z: im = z.convert("L")
    w, h = im.size; s = size / max(w, h)
    im = im.resize((max(1, round(w * s)), max(1, round(h * s))), Image.BICUBIC)
    canvas = Image.new("L", (size, size), 0); canvas.paste(im, ((size - im.width) // 2, (size - im.height) // 2))
    return canvas
def cxz_input(im): return ((TF.pil_to_tensor(im).float() - CXZ_MEAN) / CXZ_STD).expand(3, -1, -1).contiguous()
CXZ_EVAL = T.Resize((224, 224), interpolation=BICUBIC)
CXZ_AUG  = T.Compose([T.RandomResizedCrop(224, scale=(.8, 1.), ratio=(.9, 1.1), interpolation=BICUBIC),
                      T.RandomRotation(5), T.ColorJitter(brightness=.2, contrast=.2)])

class CXZDS(Dataset):
    def __init__(self, df, train=False):
        self.paths, self.y, self.train = df.resolved_path.tolist(), df[list(LABEL_COLS)].to_numpy(np.float32), train
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        im = pad_to_square(self.paths[i])
        return cxz_input(CXZ_AUG(im) if self.train else CXZ_EVAL(im)), torch.from_numpy(self.y[i])

@torch.inference_mode()
def encode_cxz(visual, df):
    visual.eval(); out = []
    for x, _ in DataLoader(CXZDS(df), batch_size=64, shuffle=False, num_workers=WORKERS):
        with torch.autocast("cuda", dtype=torch.float16): z = visual(x.to(DEVICE))
        out.append(F.normalize(z.float(), dim=-1).cpu())
    return torch.cat(out)

# ---- generic supervised pre-adaptation loop ----
def preadapt(tag, orig, trainable, extra, loader, step_fn, select_fn):
    """step_fn(student, x, y) -> (task_loss, student_emb, original_emb). Returns (best_state, history, selected_epoch)."""
    student = STUDENTS[tag]
    opt = torch.optim.AdamW([{"params": trainable, "lr": LR_BACKBONE}, {"params": extra, "lr": LR_HEAD}], weight_decay=1e-4)
    scaler = torch.amp.GradScaler("cuda")
    base = select_fn(orig)
    best = {"auc": base, "epoch": 0, "state": {k: v.detach().cpu().clone() for k, v in student.state_dict().items()}}
    hist = [{"branch": tag, "epoch": 0, "select_auroc": base}]
    print(f"[{tag}] trainable {sum(p.numel() for p in trainable + extra):,} params | SELECT AUROC before = {base:.4f}")
    for ep in range(1, ADAPT_EPOCHS + 1):
        student.train(); t0 = time.time(); L, A = [], []
        for step, (x, y) in enumerate(loader, 1):
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with torch.autocast("cuda", dtype=torch.float16):
                task, s_emb, o_emb = step_fn(student, x, y)
                a_cos = F.cosine_similarity(s_emb.float(), o_emb.float(), dim=-1).mean()
                loss = task + LAMBDA_ANCHOR * (1 - a_cos)
            if not torch.isfinite(loss): raise FloatingPointError(f"[{tag}] non-finite loss at epoch {ep} step {step}")
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(trainable + extra, 1.0); scaler.step(opt); scaler.update()
            L.append(loss.item()); A.append(a_cos.item())
            if ep == 1 and step == 1:
                print(f"  [sanity:{tag}] step 1 loss={loss.item():.4f} task={task.item():.4f} anchor_cos={a_cos.item():.4f}")
            if SMOKE and step >= SMOKE_MAX_STEPS: break
            if step % 50 == 0: print(f"  [{tag}] ep{ep} step {step}/{len(loader)} loss={np.mean(L[-50:]):.4f} | {elapsed()}")
        auc = select_fn(student)
        hist.append({"branch": tag, "epoch": ep, "select_auroc": auc, "loss": float(np.mean(L)), "anchor_cos": float(np.mean(A))})
        print(f"[{tag}] epoch {ep}/{ADAPT_EPOCHS} loss={np.mean(L):.4f} anchor_cos={np.mean(A):.4f} SELECT AUROC={auc:.4f} | {time.time()-t0:.0f}s | {elapsed()}")
        if np.isfinite(auc) and auc > best["auc"]:
            best = {"auc": auc, "epoch": ep, "state": {k: v.detach().cpu().clone() for k, v in student.state_dict().items()}}
    del opt; torch.cuda.empty_cache()
    print(f"[{tag}] selected epoch {best['epoch']} (SELECT {best['auc']:.4f})" + ("" if best["epoch"] else " -> original weights kept"))
    return best["state"], hist, best["epoch"]

def set_trainable_vit(model):
    for p in model.parameters(): p.requires_grad_(False)
    params = list(model.encoder.encoder.layers[-ADAPT_BLOCKS:].parameters()) + list(model.encoder.encoder.ln.parameters()) + list(model.head.parameters())
    for p in params: p.requires_grad_(True)
    return params

ENC = {e: {"whole": {}, "patch": {}} for e in ENCODER_SETS}      # ENC[encoder_set][branch][split] = {"zc", "logits"}
CXZ = {e: {} for e in ENCODER_SETS}                              # CXZ[encoder_set][split] = (N,512) embeddings
STUDENTS, HISTORY, SELECTED = {}, [], {}
CKPT_SHA = {"whole": file_sha256(T0_WHOLE_PATH), "patch": file_sha256(T0_PATCH_PATH), "chexzero": file_sha256(CHEXZERO_CKPT)}

# ---- whole + patch T0 branches ----
for branch, path, bs in (("whole", T0_WHOLE_PATH, BATCH_WHOLE), ("patch", T0_PATCH_PATH, BATCH_PATCH)):
    orig = load_branch(path)
    STUDENTS[branch] = copy.deepcopy(orig); trainable = set_trainable_vit(STUDENTS[branch])
    def step_fn(student, x, y, _mode=branch, _orig=orig):
        if _mode == "whole":
            out = student(x)
            with torch.no_grad(): o = _orig(x)
            return masked_bce(out["logits"], y), out["zc"], o["zc"]
        B, K = x.shape[:2]; xf = x.flatten(0, 1)
        out = student(xf)
        with torch.no_grad(): o = _orig(xf)
        return masked_bce(pool_crops(out["logits"].view(B, K, -1)), y), out["zc"], o["zc"]
    select_fn = lambda m, _mode=branch: macro_auc(YS["select"], t0_scores(encode_t0(m, COHORTS["select"], _mode)).numpy())
    loader = DataLoader(T0DS(COHORTS["preadapt"], branch, train=True), batch_size=bs, shuffle=True, num_workers=WORKERS,
                        drop_last=True, generator=torch.Generator().manual_seed(SEED))
    state, hist, SELECTED[branch] = preadapt(branch, orig, trainable, [], loader, step_fn, select_fn)
    HISTORY += hist
    adapted = copy.deepcopy(orig); adapted.load_state_dict(state); adapted.eval()
    torch.save({"model": state, "selected_epoch": SELECTED[branch], "source_ckpt_sha": CKPT_SHA[branch],
                "preadapt_patients": COHORTS["preadapt"].patient_id.astype(str).tolist()}, OUT / f"T1_CXP_{branch}_vit.pt")
    for e, m in (("original", orig), ("preadapted", adapted)):
        if e in ENCODER_SETS:
            for n in EVAL_SPLITS: ENC[e][branch][n] = encode_t0(m, COHORTS[n], branch)
    del orig, adapted, STUDENTS[branch]; torch.cuda.empty_cache()
    print(f"[{branch}] features cached for {ENCODER_SETS} x {EVAL_SPLITS} | {elapsed()}")

# ---- CheXzero ----
clip_model = load_chexzero(CHEXZERO_CKPT)
with torch.no_grad():   # no_grad (not inference_mode): W_TXT is used inside autograd during pre-adaptation
    names = [c.lower() for c in CLASSES]
    tt = F.normalize(clip_model.encode_text(open_clip.tokenize(names + [f"no {n}" for n in names]).to(DEVICE)).float(), dim=-1)
TPOS, TNEG = tt[:len(CLASSES)].cpu(), tt[len(CLASSES):].cpu()
W_TXT = (TPOS - TNEG).to(DEVICE)                                            # (5,512), frozen
cxz_orig = copy.deepcopy(clip_model.visual).eval(); del clip_model; torch.cuda.empty_cache()
def zs_scores(Z): return (100.0 * Z @ (TPOS - TNEG).T).numpy()              # zero-shot logit(pos) - logit(neg)

STUDENTS["chexzero"] = copy.deepcopy(cxz_orig)
for p in STUDENTS["chexzero"].parameters(): p.requires_grad_(False)
cxz_train = list(STUDENTS["chexzero"].transformer.resblocks[-ADAPT_BLOCKS:].parameters()) + list(STUDENTS["chexzero"].ln_post.parameters())
for p in cxz_train: p.requires_grad_(True)
cxz_bias = nn.Parameter(torch.zeros(len(CLASSES), device=DEVICE))          # calibration only; AUROC-invariant
def cxz_step(student, x, y):
    s = student(x)
    with torch.no_grad(): o = cxz_orig(x)
    return masked_bce(100.0 * F.normalize(s.float(), dim=-1) @ W_TXT.T + cxz_bias, y), s, o
cxz_select = lambda m: macro_auc(YS["select"], zs_scores(encode_cxz(m, COHORTS["select"])))
zs_sel0 = cxz_select(cxz_orig)
print(f"[chexzero] zero-shot SELECT AUROC (original) = {zs_sel0:.4f}")
if zs_sel0 < 0.70: print("WARNING: low CheXzero zero-shot -- check checkpoint/preprocessing before trusting the prior.")
cxz_loader = DataLoader(CXZDS(COHORTS["preadapt"], train=True), batch_size=BATCH_CXZ, shuffle=True, num_workers=WORKERS,
                        drop_last=True, generator=torch.Generator().manual_seed(SEED))
state, hist, SELECTED["chexzero"] = preadapt("chexzero", cxz_orig, cxz_train, [cxz_bias], cxz_loader, cxz_step, cxz_select)
HISTORY += hist
cxz_adapted = copy.deepcopy(cxz_orig); cxz_adapted.load_state_dict(state); cxz_adapted.eval()
torch.save({"visual_state": state, "selected_epoch": SELECTED["chexzero"], "source_ckpt_sha": CKPT_SHA["chexzero"]}, OUT / "T1_CXP_chexzero_visual.pt")
for e, m in (("original", cxz_orig), ("preadapted", cxz_adapted)):
    if e in ENCODER_SETS:
        for n in EVAL_SPLITS: CXZ[e][n] = encode_cxz(m, COHORTS[n])
del cxz_orig, cxz_adapted, STUDENTS["chexzero"]; torch.cuda.empty_cache()
pd.DataFrame(HISTORY).to_csv(OUT / "preadapt_history.csv", index=False)
print("Selected pre-adaptation epochs:", SELECTED, "| Cell 2 ready.", elapsed())

[whole] trainable 14,443,141 params | SELECT AUROC before = 0.5832
  [sanity:whole] step 1 loss=1.2899 task=1.2889 anchor_cos=0.9901
  [whole] ep1 step 50/50 loss=0.8370 | t+1.4 min
[whole] epoch 1/5 loss=0.8370 anchor_cos=0.5030 SELECT AUROC=0.7425 | 37s | t+1.5 min
  [whole] ep2 step 50/50 loss=0.6294 | t+1.9 min
[whole] epoch 2/5 loss=0.6294 anchor_cos=0.3438 SELECT AUROC=0.7656 | 27s | t+1.9 min
  [whole] ep3 step 50/50 loss=0.6050 | t+2.4 min
[whole] epoch 3/5 loss=0.6050 anchor_cos=0.3136 SELECT AUROC=0.7715 | 29s | t+2.4 min
  [whole] ep4 step 50/50 loss=0.5819 | t+2.9 min
[whole] epoch 4/5 loss=0.5819 anchor_cos=0.3455 SELECT AUROC=0.7815 | 30s | t+2.9 min
  [whole] ep5 step 50/50 loss=0.5695 | t+3.4 min
[whole] epoch 5/5 loss=0.5695 anchor_cos=0.3433 SELECT AUROC=0.7863 | 29s | t+3.4 min
[whole] selected epoch 5 (SELECT 0.7863)
[whole] features cached for ('original', 'preadapted') x ['support', 'select', 'dev', 'expert_valid'] | t+4.3 min
[patch] trainable 14,443,141 params |

In [3]:
# =============================================================================
# Cell 3/4 — informed class-wise routing (whole <-> patch) + routed features + audit
#   ROUTING_SOURCE = "nih_val_csv": per-class weights from the checkpoints' own NIH
#     validation AUROC (per_class_validation_metrics.csv, tier T0). Q1 rule: each
#     branch counts by its advantage over chance (AUROC - 0.5, zeroed inside the
#     deadband); patch weight = adv_patch / (adv_patch + adv_whole). Same weights for
#     original and pre-adapted encoders (they come from the source checkpoints).
#   ROUTING_SOURCE = "select": Q1's rule refit on CheXpert SELECT per encoder set.
#   routed feature per class = normalize(w_p * patch_c + w_w * whole)
#     whole_c = whole zc (128), patch_c = six-crop zc attention-pooled with class-c crop logits (Q1)
# =============================================================================
def advantage_weight(a_whole, a_patch, deadband=ROUTING_DEADBAND):
    adv_w = (a_whole - .5) if (a_whole - .5) > deadband else 0.0
    adv_p = (a_patch - .5) if (a_patch - .5) > deadband else 0.0
    return adv_p / (adv_p + adv_w) if (adv_p + adv_w) > 1e-6 else 0.5

def routing_from_csv(path):
    df = pd.read_csv(path)
    df = df[df["tier"].astype(str).eq("T0") & df["stage"].astype(str).str.contains("validation", case=False)]
    piv = df.pivot_table(index="pathology", columns="branch", values="auroc")
    missing = [c for c in CLASSES if c not in piv.index]
    if missing or not {"whole", "patch"} <= set(piv.columns): raise ValueError(f"routing CSV lacks {missing or 'whole/patch columns'}")
    rows = [{"class": c, "auroc_whole": piv.loc[c, "whole"], "auroc_patch": piv.loc[c, "patch"],
             "patch_weight": advantage_weight(piv.loc[c, "whole"], piv.loc[c, "patch"])} for c in CLASSES]
    return pd.DataFrame(rows)

def routing_from_select(enc):
    y = YS["select"]; w = ENC[enc]["whole"]["select"]["logits"].numpy()
    p = pool_crops(ENC[enc]["patch"]["select"]["logits"]).numpy()
    rows = []
    for c, name in enumerate(CLASSES):
        m = np.isfinite(y[:, c]); ok = m.sum() and len(np.unique(y[m, c])) == 2
        aw = roc_auc_score(y[m, c], w[m, c]) if ok else .5; ap = roc_auc_score(y[m, c], p[m, c]) if ok else .5
        rows.append({"class": name, "auroc_whole": aw, "auroc_patch": ap, "patch_weight": advantage_weight(aw, ap)})
    return pd.DataFrame(rows)

ROUTING = {}
for enc in ENCODER_SETS:
    if ROUTING_SOURCE == "nih_val_csv": ROUTING[enc] = routing_from_csv(ROUTING_CSV)
    elif ROUTING_SOURCE == "select":    ROUTING[enc] = routing_from_select(enc)
    elif ROUTING_SOURCE == "equal":     ROUTING[enc] = pd.DataFrame({"class": CLASSES, "patch_weight": [.5] * len(CLASSES)})
    else: raise ValueError(ROUTING_SOURCE)
    print(f"\n=== ROUTING [{enc}] source={ROUTING_SOURCE} deadband={ROUTING_DEADBAND} ===\n" + ROUTING[enc].round(3).to_string(index=False))
PATCH_W = {e: torch.tensor(ROUTING[e].patch_weight.to_numpy(), dtype=torch.float32) for e in ENCODER_SETS}

def routed(enc, split):
    """(N,5,128) routed features. Ops run outside inference_mode -> normal (autograd-safe) tensors."""
    p, w = ENC[enc]["patch"][split], ENC[enc]["whole"][split]
    whole_feat = w["zc"].clone()[:, None, :]                                              # (N,1,128)
    attn = torch.softmax(p["logits"].clone(), dim=1)                                      # (N,6,5)
    patch_feat = torch.einsum("nkc,nkd->ncd", attn, p["zc"].clone())                     # (N,5,128)
    pw = PATCH_W[enc][None, :, None]
    return F.normalize(pw * patch_feat + (1 - pw) * whole_feat, dim=-1)

FEATS = {e: {n: routed(e, n) for n in EVAL_SPLITS} for e in ENCODER_SETS}

# Frozen audit (no few-shot training): each branch, the routed logit mix, and the CheXzero zero-shot prior
audit = []
for e, n in product(ENCODER_SETS, [s for s in EVAL_SPLITS if s != "support"]):
    y = YS[n]; ws = ENC[e]["whole"][n]["logits"].numpy()
    ps = pool_crops(ENC[e]["patch"][n]["logits"]).numpy()
    pw = PATCH_W[e].numpy()
    audit.append({"encoder": e, "split": n, "whole": macro_auc(y, ws), "patch": macro_auc(y, ps),
                  "routed_mix": macro_auc(y, pw * ps + (1 - pw) * ws), "chexzero_zs": macro_auc(y, zs_scores(CXZ[e][n]))})
AUDIT = pd.DataFrame(audit); AUDIT.to_csv(OUT / "frozen_audit.csv", index=False)
print("\n=== FROZEN AUDIT, macro AUROC (before any few-shot training) ===\n" + AUDIT.round(4).to_string(index=False))
print("Reference, Q1 NIH-only frozen on DEV: whole 0.589 | patch 0.501 | routed 0.592")
print("Cell 3 ready.", elapsed())


=== ROUTING [original] source=nih_val_csv deadband=0.03 ===
           class  auroc_whole  auroc_patch  patch_weight
     Atelectasis        0.781        0.790         0.508
    Cardiomegaly        0.857        0.821         0.474
   Consolidation        0.691        0.698         0.509
           Edema        0.843        0.786         0.454
Pleural Effusion        0.777        0.826         0.541

=== ROUTING [preadapted] source=nih_val_csv deadband=0.03 ===
           class  auroc_whole  auroc_patch  patch_weight
     Atelectasis        0.781        0.790         0.508
    Cardiomegaly        0.857        0.821         0.474
   Consolidation        0.691        0.698         0.509
           Edema        0.843        0.786         0.454
Pleural Effusion        0.777        0.826         0.541

=== FROZEN AUDIT, macro AUROC (before any few-shot training) ===
   encoder        split  whole  patch  routed_mix  chexzero_zs
  original       select 0.5832 0.4876      0.5812       0.8929


In [4]:
# =============================================================================
# Cell 4/4 — QUANTUM FEW-SHOT HEADS (Q1, unchanged), FINE-TUNED WITH LABELS
#   Input  : routed (N,5,128) features from Cell 3 (whole <-> patch, informed routing),
#            for the ORIGINAL and the PRE-ADAPTED encoders
#   Prior  : RESIDUAL_PRIOR="chexzero" -> logit = CheXzero zero-shot logit (same encoder set)
#            + head output, identical for every head. "none" -> exact Q1 heads, no prior.
#   Heads  : H0 classical per-class MLPs (parameter-matched to H1 / H2, + wide control)
#            H1 quantum: one VQC per pathology, N_QUBITS x VQC_LAYERS, data re-uploading
#               RY/RZ(x) -> RY/RZ(theta), ring of CNOTs, <Z> readout, exact torch statevector
#            H2 hybrid: H0 + learned per-class gate towards H1
#            With 128-d input these are Q1's exact heads (same parameter counts as Q1).
#   Support: K pos + K neg LABELLED patients per class from the 'support' cohort only
#            (never from the 1.6k pre-adaptation cohort). Epoch budget tuned on SELECT.
#   Not included (vs Q1): NIH replay / NIH retention scoring.
# Quantum execution guarantee:
#   * Every H1/H2 circuit evaluation (training, meta-validation, prediction) is a PennyLane QNode
#     on qml.device("default.qubit") -- a statevector SIMULATOR, not quantum hardware.
#   * The hand-written torch simulator is used ONLY inside the equivalence check; any other call
#     raises. Every run records its QNode call count; a quantum run with 0 QNode calls, or any
#     torch-simulator call outside the check, aborts the notebook.
#   * The backend, PennyLane version and device go into the run fingerprint and every saved run,
#     so results from an older torch-backend run can never be silently reused.
# Baselines: P0_prior_only (CheXzero zero-shot, no training) is reported and tested against.
# =============================================================================

# ---- quantum / training settings (Q1 values unless noted) ----
N_QUBITS, VQC_LAYERS = (4, 1) if SMOKE else (6, 3)
QUANTUM_BACKEND   = "pennylane"            # locked: all H1/H2 circuits run as PennyLane QNodes
PL_DEVICE_NAME    = "default.qubit"        # PennyLane statevector simulator
PENNYLANE_VERSION = "0.42.3"
assert QUANTUM_BACKEND == "pennylane", "This notebook is locked to the PennyLane backend."
LR, WEIGHT_DECAY, GRAD_CLIP, RANK_WEIGHT = 0.01, 1e-3, 5.0, 0.30
LR_GATE         = 0.10                      # hybrid gate only (no weight decay); at LR it never left its init
RESIDUAL_PRIOR  = "chexzero"                # "none" -> exact Q1 setting (no zero-shot prior)
Q_SHOTS         = (1,) if SMOKE else (1, 5)
Q_EPISODES      = 2 if SMOKE else 10
MODEL_SEEDS     = 1 if SMOKE else 3          # init-seed variance, averaged per episode before tests
Q_ENCODERS      = ENCODER_SETS
EPOCH_SELECTION = "fixed" if SMOKE else "meta_val"
EPOCH_GRID      = (1, 2) if SMOKE else (1, 2, 5, 10, 25, 50, 100, 150)   # small budgets: prior is strong
N_TUNE_EPISODES = 1 if SMOKE else 5
FIXED_EPOCHS    = 2 if SMOKE else 150
PERMUTATIONS    = 100 if SMOKE else 5000
MAX_RUNTIME_HOURS = 11.0
IN_DIM = 128                                # routed ViT features (Q1)
Q_TEST_SPLITS = [s for s in ("dev", "expert_valid") if s in COHORTS]

try:
    import pennylane as qml
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"pennylane=={PENNYLANE_VERSION}"], check=True)
    import pennylane as qml
print(f"PennyLane {qml.__version__} | device {PL_DEVICE_NAME} (statevector simulator) | interface torch, diff_method backprop")
if qml.__version__ != PENNYLANE_VERSION: print(f"NOTE: expected PennyLane {PENNYLANE_VERSION}, found {qml.__version__} (equivalence check below still gates the run)")

# ---- execution ledger: proves which simulator produced every number ----
QSTATS = {"qnode_calls": 0, "qnode_samples": 0, "torch_sim_calls": 0}
_TORCH_REFERENCE = {"allowed": False}

# ----------------------------------------------------------------------------
# 1. Losses (NaN labels ignored; masked_bce defined in Cell 1)
# ----------------------------------------------------------------------------
def rank_loss(scores, y):
    losses = []
    for c in range(len(CLASSES)):
        known = torch.isfinite(y[:, c])
        pos, neg = scores[known & (y[:, c] >= .5), c], scores[known & (y[:, c] < .5), c]
        if len(pos) and len(neg): losses.append(F.softplus(-(pos[:, None] - neg[None, :])).mean())
    return torch.stack(losses).mean() if losses else scores.sum() * 0

def objective(logits, y): return masked_bce(logits, y) + RANK_WEIGHT * rank_loss(logits, y)

# ----------------------------------------------------------------------------
# 2. Heads (verbatim Q1)
# ----------------------------------------------------------------------------
class PerClassMLP(nn.Module):
    def __init__(self, hidden, in_dim=IN_DIM):
        super().__init__()
        self.branches = nn.ModuleList([nn.Sequential(
            nn.Linear(in_dim, hidden), nn.GELU(), nn.LayerNorm(hidden),
            nn.Linear(hidden, hidden // 2), nn.GELU(), nn.LayerNorm(hidden // 2),
            nn.Linear(hidden // 2, 1)) for _ in CLASSES])
    def forward(self, x):                                    # x: (N,5,D) -> (N,5)
        return torch.cat([branch(x[:, c]) for c, branch in enumerate(self.branches)], dim=1)

def ry(t):
    c, s = torch.cos(t / 2), torch.sin(t / 2)
    return torch.stack([c, -s, s, c], -1).reshape(t.shape + (2, 2)).to(torch.complex64)

def rz(t):
    zero = torch.zeros_like(t)
    a, b = torch.complex(torch.cos(-t / 2), torch.sin(-t / 2)), torch.complex(torch.cos(t / 2), torch.sin(t / 2))
    return torch.stack([a, torch.complex(zero, zero), torch.complex(zero, zero), b], -1).reshape(t.shape + (2, 2))

def apply_1q(state, U, q, nq):
    B = len(state)
    x = state.view((B,) + (2,) * nq).movedim(q + 1, -1)
    return torch.einsum("b...j,bij->b...i", x, U).movedim(-1, q + 1).reshape(B, -1)

_CNOT_PERM = {}
def apply_cnot(state, control, target, nq):
    key = (control, target, nq, state.device)
    if key not in _CNOT_PERM:
        _CNOT_PERM[key] = torch.tensor([i ^ (1 << (nq-1-target)) if (i >> (nq-1-control)) & 1 else i
                                        for i in range(2 ** nq)], device=state.device)
    return state[:, _CNOT_PERM[key]]

def z_expectations(state, nq):
    probs = state.abs().square().view((len(state),) + (2,) * nq)
    out = []
    for q in range(nq):
        marg = probs.sum(dim=tuple(d + 1 for d in range(nq) if d != q))
        out.append(marg[:, 0] - marg[:, 1])
    return torch.stack(out, 1)

def feature_index(layer, q, nq, axis, n_features):
    return (2 * layer * nq + q + axis * nq) % n_features

class PathologyVQC(nn.Module):
    def __init__(self, nq=N_QUBITS, layers=VQC_LAYERS):
        super().__init__()
        self.nq, self.layers = nq, layers
        self.theta = nn.Parameter(torch.randn(layers, nq, 2) * .05)
        self._qnode = None
    def forward(self, x, backend=None):                      # x: (B, 2*nq) in [-1, 1]
        backend = backend or QUANTUM_BACKEND
        if backend == "pennylane": return self._forward_pennylane(x)
        if backend != "torch": raise ValueError(backend)
        if not _TORCH_REFERENCE["allowed"]:
            raise RuntimeError("torch simulator called outside the equivalence check -- PennyLane lock violated")
        QSTATS["torch_sim_calls"] += 1
        B, nq, xa = len(x), self.nq, x * math.pi
        state = torch.zeros(B, 2 ** nq, device=x.device, dtype=torch.complex64); state[:, 0] = 1
        for layer in range(self.layers):
            for q in range(nq):
                state = apply_1q(state, ry(xa[:, feature_index(layer, q, nq, 0, xa.shape[1])]), q, nq)
                state = apply_1q(state, rz(xa[:, feature_index(layer, q, nq, 1, xa.shape[1])]), q, nq)
                state = apply_1q(state, ry(self.theta[layer, q, 0].expand(B)), q, nq)
                state = apply_1q(state, rz(self.theta[layer, q, 1].expand(B)), q, nq)
            for q in range(nq): state = apply_cnot(state, q, (q + 1) % nq, nq)
        return z_expectations(F.normalize(state, dim=1), nq)
    def _forward_pennylane(self, x):
        if self._qnode is None:
            nq, layers = self.nq, self.layers
            def circuit(xa, theta):
                for layer in range(layers):
                    for q in range(nq):
                        qml.RY(xa[..., feature_index(layer, q, nq, 0, xa.shape[-1])], wires=q)
                        qml.RZ(xa[..., feature_index(layer, q, nq, 1, xa.shape[-1])], wires=q)
                        qml.RY(theta[layer, q, 0], wires=q); qml.RZ(theta[layer, q, 1], wires=q)
                    for q in range(nq): qml.CNOT(wires=[q, (q + 1) % nq])
                return [qml.expval(qml.PauliZ(w)) for w in range(nq)]
            self._qnode = qml.QNode(circuit, qml.device(PL_DEVICE_NAME, wires=nq), interface="torch", diff_method="backprop")
        # The QNode runs on CPU (avoids CPU/CUDA mixing inside default.qubit); .cpu()/.to() are
        # differentiable, so gradients flow back to the GPU encoders and to theta.
        out = self._qnode((x * math.pi).cpu(), self.theta.cpu())
        QSTATS["qnode_calls"] += 1; QSTATS["qnode_samples"] += len(x)
        return torch.stack(list(out), -1).to(device=x.device, dtype=x.dtype)

class QuantumHead(nn.Module):
    def __init__(self, in_dim=IN_DIM, nq=N_QUBITS):
        super().__init__()
        self.encoders = nn.ModuleList([nn.Linear(in_dim, 2 * nq) for _ in CLASSES])   # 128 -> 2*nq angles
        self.circuits = nn.ModuleList([PathologyVQC(nq) for _ in CLASSES])
        self.readouts = nn.ModuleList([nn.Linear(nq, 1) for _ in CLASSES])
    def forward(self, x, backend=None):
        return torch.cat([self.readouts[c](self.circuits[c](torch.tanh(self.encoders[c](x[:, c])), backend))
                          for c in range(len(CLASSES))], dim=1)

class HybridHead(nn.Module):
    def __init__(self, classical, quantum):
        super().__init__()
        self.classical, self.quantum = classical, quantum
        self.gate_raw = nn.Parameter(torch.full((len(CLASSES),), -0.5))
    def gate(self): return torch.sigmoid(self.gate_raw)
    def forward(self, x):
        c0, c1 = self.classical(x), self.quantum(x)
        return c0 + self.gate()[None] * (c1 - c0)

def n_params(model): return sum(p.numel() for p in model.parameters())
def matched_hidden(target): return min(range(4, 129), key=lambda h: (abs(n_params(PerClassMLP(h)) - target), h))
H_MATCH_Q = matched_hidden(n_params(QuantumHead()))
H_MATCH_HYBRID = matched_hidden(n_params(HybridHead(PerClassMLP(H_MATCH_Q), QuantumHead())))
H_WIDE = 48
HEADS = {
    "H0_classical_matchQ":      lambda: PerClassMLP(H_MATCH_Q),
    "H0_classical_matchHybrid": lambda: PerClassMLP(H_MATCH_HYBRID),
    "H0_classical_wide":        lambda: PerClassMLP(H_WIDE),
    "H1_quantum":               lambda: QuantumHead(),
    "H2_hybrid":                lambda: HybridHead(PerClassMLP(H_MATCH_Q), QuantumHead()),
}
PRIMARY_TESTS   = [("H1_quantum", "H0_classical_matchQ"), ("H2_hybrid", "H0_classical_matchHybrid")]
SECONDARY_TESTS = [("H1_quantum", "H0_classical_wide")]
QUANTUM_HEADS   = {"H1_quantum", "H2_hybrid"}
PRIOR_HEAD      = "P0_prior_only"
PRIOR_TESTS     = [(h, PRIOR_HEAD) for h in HEADS]          # does few-shot training beat the untrained prior?
PARAMS = {n: n_params(m()) for n, m in HEADS.items()}
print("Parameters:", {k: f"{v:,}" for k, v in PARAMS.items()}, f"| qubits={N_QUBITS} layers={VQC_LAYERS} backend={QUANTUM_BACKEND}")
for method, control in PRIMARY_TESTS:
    ratio = PARAMS[control] / PARAMS[method]
    if not 0.9 <= ratio <= 1.1: raise RuntimeError(f"Capacity match failed: {control}/{method} = {ratio:.2f}")
    print(f"  {control} / {method} parameter ratio = {ratio:.3f}")

# ----------------------------------------------------------------------------
# 3. Inputs: routed (N,5,128) features + CheXzero zero-shot prior per encoder set / split
# ----------------------------------------------------------------------------
def qdata(enc, split):
    X = FEATS[enc][split].to(DEVICE)
    prior = torch.tensor(zs_scores(CXZ[enc][split]), dtype=torch.float32, device=DEVICE) if RESIDUAL_PRIOR == "chexzero" \
            else torch.zeros(len(X), len(CLASSES), device=DEVICE)
    return X, prior
QX = {enc: {s: qdata(enc, s) for s in EVAL_SPLITS} for enc in Q_ENCODERS}
Y_POOL, Y_SEL = YS["support"], YS["select"]

# ---- mandatory PennyLane gate: equivalence + circuit diagram + gradient check ----
seed_all(SEED); probe = QuantumHead().to(DEVICE); xp = QX[Q_ENCODERS[0]]["select"][0][:8]
with torch.no_grad():
    y_pl = probe(xp, "pennylane")
    _TORCH_REFERENCE["allowed"] = True
    try:     y_tq = probe(xp, "torch")
    finally: _TORCH_REFERENCE["allowed"] = False
gap = float((y_tq - y_pl).abs().max())
print(f"[equivalence] torch reference vs PennyLane {PL_DEVICE_NAME}: max |Δlogit| = {gap:.2e}")
if gap > 1e-4: raise RuntimeError("Quantum simulator disagrees with PennyLane -- do not trust H1/H2.")
vqc0 = probe.circuits[0]
with torch.no_grad(): ang = torch.tanh(probe.encoders[0](xp[0, 0])).cpu() * math.pi   # one sample -> numeric angles in the drawing
print("[circuit] class 0 VQC as executed by PennyLane:\n" + qml.draw(vqc0._qnode, decimals=2)(ang, vqc0.theta.detach().cpu()))
probe.zero_grad(); probe(xp).sum().backward()
g = probe.circuits[0].theta.grad
if g is None or not torch.isfinite(g).all() or float(g.abs().sum()) == 0:
    raise RuntimeError("No gradient reached the circuit parameters through the PennyLane QNode.")
print(f"[gradient] |dL/dtheta| through the QNode = {float(g.abs().sum()):.3e} (non-zero, finite)")
del probe, vqc0
QSTATS.update(qnode_calls=0, qnode_samples=0, torch_sim_calls=0)  # discard all one-time gate calls; ledger counts real runs only

# ----------------------------------------------------------------------------
# 4. Labelled K-shot support (label budget enforced, exactly as Q1)
# ----------------------------------------------------------------------------
def sample_support(Y, k, rng):
    Y = np.asarray(Y, float); masked, rows = np.full_like(Y, np.nan), []
    for c in range(len(CLASSES)):
        known = np.isfinite(Y[:, c])
        pos, neg = np.flatnonzero(known & (Y[:, c] >= .5)), np.flatnonzero(known & (Y[:, c] < .5))
        if min(len(pos), len(neg)) < k: raise RuntimeError(f"{CLASSES[c]}: pool too small for {k}-shot -- raise CAP")
        for i in np.concatenate([rng.choice(pos, k, replace=False), rng.choice(neg, k, replace=False)]):
            masked[i, c] = Y[i, c]; rows.append(i)
    idx = np.array(sorted(set(rows)))
    assert all((masked[idx, c] == 1).sum() == k and (masked[idx, c] == 0).sum() == k for c in range(len(CLASSES)))
    return idx, torch.tensor(masked[idx], dtype=torch.float32, device=DEVICE)

def support_rng(k, episode, tuning=False):
    return np.random.default_rng(SEED + 7919 * int(k) + 101 * episode + (500_000 if tuning else 0))

# ----------------------------------------------------------------------------
# 5. Fine-tuning: full-batch AdamW on the labelled support (no replay)
# ----------------------------------------------------------------------------
def train_head(make, seed, x, prior, y, epochs, eval_epochs=(), x_eval=None, prior_eval=None, y_eval=None):
    seed_all(seed); model = make().to(DEVICE)
    gate = [p for n, p in model.named_parameters() if n == "gate_raw"]
    rest = [p for n, p in model.named_parameters() if n != "gate_raw"]
    groups = [{"params": rest}] + ([{"params": gate, "lr": LR_GATE, "weight_decay": 0.0}] if gate else [])
    opt = torch.optim.AdamW(groups, lr=LR, weight_decay=WEIGHT_DECAY); curve = {}
    for epoch in range(1, epochs + 1):
        model.train(); opt.zero_grad()
        loss = objective(model(x) + prior, y)
        if not torch.isfinite(loss): raise FloatingPointError(f"non-finite loss (seed {seed}, epoch {epoch})")
        loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP); opt.step()
        if epoch in eval_epochs:
            model.eval()
            with torch.inference_mode(): curve[epoch] = macro_auc(y_eval, (model(x_eval) + prior_eval).cpu().numpy())
    return model.eval(), {"epochs": epochs, "final_loss": loss.item()}, curve

@torch.inference_mode()
def predict(model, enc):
    return {s: (model(QX[enc][s][0]) + QX[enc][s][1]).cpu().numpy() for s in Q_TEST_SPLITS}

QFP = sha_of({"ckpt": CKPT_SHA, "preadapt_selected": SELECTED, "preadapt_cfg": [ADAPT_EPOCHS, ADAPT_BLOCKS, LR_BACKBONE, LR_HEAD, LAMBDA_ANCHOR],
              "routing": {e: PATCH_W[e].tolist() for e in Q_ENCODERS}, "routing_source": ROUTING_SOURCE,
              "cohorts": sha_of({k: v.patient_id.astype(str).tolist() for k, v in COHORTS.items()}), "label_policy": LABEL_POLICY,
              "quantum": [N_QUBITS, VQC_LAYERS, QUANTUM_BACKEND, PL_DEVICE_NAME, qml.__version__], "pooling": PATCH_POOLING, "lr_gate": LR_GATE, "optim": [LR, WEIGHT_DECAY, GRAD_CLIP, RANK_WEIGHT],
              "prior": RESIDUAL_PRIOR, "hidden": [H_MATCH_Q, H_MATCH_HYBRID, H_WIDE], "run_mode": RUN_MODE, "encoders": list(Q_ENCODERS),
              "epochs": [EPOCH_SELECTION, EPOCH_GRID, N_TUNE_EPISODES, FIXED_EPOCHS], "shots": list(Q_SHOTS), "seed": SEED})[:12]
QDIR = OUT / f"quantum_run_{QFP}"; QDIR.mkdir(parents=True, exist_ok=True)
print("Quantum run folder:", QDIR)

# ---- 1-batch sanity: every head trains 2 epochs, finite and non-constant (head output itself, prior excluded)
idx, ys = sample_support(Y_POOL, Q_SHOTS[0], support_rng(Q_SHOTS[0], 0))
x_s, p_s = QX[Q_ENCODERS[0]]["support"][0][idx], QX[Q_ENCODERS[0]]["support"][1][idx]
for name, make in HEADS.items():
    m, _, _ = train_head(make, SEED, x_s, p_s, ys, epochs=2)
    with torch.inference_mode(): out = m(QX[Q_ENCODERS[0]]["dev"][0]).cpu().numpy()
    assert np.isfinite(out).all() and not np.allclose(out, out[0], atol=1e-6), f"{name}: bad or constant output"
print(f"[sanity] {Q_SHOTS[0]}-shot support = {len(idx)} labelled patients; all heads finite and non-constant. {elapsed()}")

# ----------------------------------------------------------------------------
# 6. Epoch budget per (encoder, shot, head): meta-validation on SELECT-only K-shot tasks
# ----------------------------------------------------------------------------
budget_file = QDIR / "epoch_budget.json"
if EPOCH_SELECTION == "fixed":
    EPOCHS_FOR = {(e, str(k), n): FIXED_EPOCHS for e in Q_ENCODERS for k in Q_SHOTS for n in HEADS}
elif budget_file.is_file():
    EPOCHS_FOR = {tuple(key.split("|")): v for key, v in json.loads(budget_file.read_text()).items()}
    print("Restored epoch budget.")
else:
    rows_mv = []
    for enc, k in product(Q_ENCODERS, Q_SHOTS):
        Xs, Ps = QX[enc]["select"]
        for te in range(N_TUNE_EPISODES):
            idx, ys = sample_support(Y_SEL, k, support_rng(k, te, tuning=True))
            query = np.setdiff1d(np.arange(len(Y_SEL)), idx)
            for hi, (name, make) in enumerate(HEADS.items()):
                _, _, curve = train_head(make, SEED + 500_000 + 1000 * te + 17 * hi, Xs[idx], Ps[idx], ys, max(EPOCH_GRID),
                                         eval_epochs=EPOCH_GRID, x_eval=Xs[query], prior_eval=Ps[query], y_eval=Y_SEL[query])
                rows_mv += [{"encoder": enc, "shot": str(k), "head": name, "tune_episode": te, "epochs": e, "auroc": a} for e, a in curve.items()]
            print(f"[meta-val] {enc} | {k}-shot tuning episode {te + 1}/{N_TUNE_EPISODES} | {elapsed()}")
    curves = pd.DataFrame(rows_mv); curves.to_csv(QDIR / "epoch_budget_curves.csv", index=False)
    mean_curve = curves.groupby(["encoder", "shot", "head", "epochs"], as_index=False).auroc.mean()
    EPOCHS_FOR = {}
    for (enc, k, name), g in mean_curve.groupby(["encoder", "shot", "head"]):
        EPOCHS_FOR[(enc, k, name)] = int(g.sort_values(["auroc", "epochs"], ascending=[False, True]).epochs.iloc[0])
    budget_file.write_text(json.dumps({f"{e}|{k}|{n}": v for (e, k, n), v in EPOCHS_FOR.items()}, indent=1))
    print("\n=== META-VALIDATION (mean SELECT macro AUROC by epoch budget) ===\n"
          + mean_curve.pivot_table(index=["encoder", "shot", "head"], columns="epochs", values="auroc").round(4).to_string())

# ----------------------------------------------------------------------------
# 7. Sweep: encoder x shot x episode x seed x head (resumable; support shared -> paired)
# ----------------------------------------------------------------------------
Q_ROWS, Q_LOG = [], []
for k in Q_SHOTS:
    for ep in range(Q_EPISODES):
        idx, ys = sample_support(Y_POOL, k, support_rng(k, ep))
        for enc in Q_ENCODERS:
            Xt, Pt = QX[enc]["support"]
            for ms in range(MODEL_SEEDS):
                for hi, (name, make) in enumerate(HEADS.items()):
                    E = EPOCHS_FOR[(enc, str(k), name)]
                    path = QDIR / f"{enc}_shot{k}_ep{ep}_seed{ms}_{name}_E{E}.pt"
                    is_quantum = name in QUANTUM_HEADS
                    if path.is_file():
                        saved = torch.load(path, map_location="cpu", weights_only=False)
                        if not np.array_equal(saved["support_idx"], idx): raise RuntimeError(f"support mismatch in {path.name}")
                        preds, meta = saved["preds"], saved["meta"]
                        if is_quantum and not (meta.get("backend") == "pennylane" and meta.get("qnode_calls", 0) > 0):
                            raise RuntimeError(f"{path.name}: saved run has no PennyLane provenance -- delete it and re-run")
                    else:
                        if (time.time() - RUN_T0) / 3600 > MAX_RUNTIME_HOURS:
                            raise RuntimeError("MAX_RUNTIME_HOURS reached -- re-run this cell to resume from saved runs.")
                        t0 = time.time(); q0, t_before = QSTATS["qnode_calls"], QSTATS["torch_sim_calls"]
                        model, meta, _ = train_head(make, SEED + 1000 * ep + 100_000 * ms + 17 * hi, Xt[idx], Pt[idx], ys, E)
                        preds = predict(model, enc)
                        meta["qnode_calls"] = QSTATS["qnode_calls"] - q0
                        meta["backend"] = "pennylane" if is_quantum else "classical"
                        if is_quantum: meta.update(pl_device=PL_DEVICE_NAME, pl_version=qml.__version__)
                        if QSTATS["torch_sim_calls"] != t_before: raise RuntimeError("torch simulator used in a real run")
                        if is_quantum and meta["qnode_calls"] == 0: raise RuntimeError(f"{name}: no PennyLane QNode executed")
                        if not is_quantum and meta["qnode_calls"] != 0: raise RuntimeError(f"{name}: classical head touched a QNode")
                        if isinstance(model, HybridHead): meta["gate"] = model.gate().detach().cpu().round(decimals=3).tolist()
                        meta["seconds"] = round(time.time() - t0, 1)
                        torch.save({"fingerprint": QFP, "preds": preds, "meta": meta, "support_idx": idx}, path)
                    for s in Q_TEST_SPLITS:
                        m = macro_metrics(YS[s], preds[s])
                        Q_ROWS.append({"encoder": enc, "shot": k, "episode": ep, "seed": ms, "head": name, "split": s,
                                       "macro_auroc": m["macro_auroc"], "macro_auprc": m["macro_auprc"],
                                       **{f"auroc_{c}": a for c, (a, _) in m["per_class"].items()}})
                    Q_LOG.append({"encoder": enc, "shot": k, "episode": ep, "seed": ms, "head": name, "n_support": len(idx), **meta})
            for s in Q_TEST_SPLITS:   # P0: CheXzero zero-shot prior alone, no training (same for every episode)
                m = macro_metrics(YS[s], QX[enc][s][1].cpu().numpy())
                Q_ROWS.append({"encoder": enc, "shot": k, "episode": ep, "seed": 0, "head": PRIOR_HEAD, "split": s,
                               "macro_auroc": m["macro_auroc"], "macro_auprc": m["macro_auprc"],
                               **{f"auroc_{c}": a for c, (a, _) in m["per_class"].items()}})
        dev_now = pd.DataFrame(Q_ROWS).query("shot == @k and episode == @ep and split == 'dev'")
        print(f"{k}-shot ep {ep + 1}/{Q_EPISODES} | DEV macro AUROC: "
              + ", ".join(f"{r.encoder[:4]}/{r.head}={r.macro_auroc:.3f}" for r in dev_now.itertuples()) + f" | {elapsed()}")
pd.DataFrame(Q_LOG).to_csv(QDIR / "run_log.csv", index=False)

# ----------------------------------------------------------------------------
# 8. Report: mean ± 95% t-CI over episodes; paired sign-flip permutation tests
# ----------------------------------------------------------------------------
qdf = pd.DataFrame(Q_ROWS).groupby(["encoder", "shot", "episode", "head", "split"], as_index=False).mean(numeric_only=True)
qdf.to_csv(QDIR / "per_episode_metrics.csv", index=False)

def ci95(v):
    v = np.asarray(v, float); v = v[np.isfinite(v)]
    return float(student_t.ppf(.975, len(v) - 1) * v.std(ddof=1) / np.sqrt(len(v))) if len(v) > 1 else np.nan

def signflip_p(d, n_perm=PERMUTATIONS, seed=SEED):
    d = np.asarray(d, float); d = d[np.isfinite(d)]; obs = abs(d.mean())
    if len(d) <= 12:   # exact: enumerate every sign pattern
        signs = np.array(list(product((1, -1), repeat=len(d))))
    else:
        signs = np.random.default_rng(seed).choice((1, -1), size=(n_perm, len(d)))
    return float((np.abs((signs * d).mean(1)) >= obs - 1e-12).mean())

def holm(ps):
    order = np.argsort(ps); adj = np.empty(len(ps)); run = 0.0
    for r, i in enumerate(order):
        run = max(run, (len(ps) - r) * ps[i]); adj[i] = min(1.0, run)
    return adj

summ = (qdf.groupby(["split", "shot", "encoder", "head"]).macro_auroc
        .agg(mean="mean", ci95=ci95, n="size").reset_index())
summ.to_csv(QDIR / "summary_quantum.csv", index=False)
print("\n=== LABELLED K-SHOT FINE-TUNING (macro AUROC, mean ± 95% CI over episodes) ===\n" + summ.round(4).to_string(index=False))
print(f"(RESIDUAL_PRIOR={RESIDUAL_PRIOR}. No-training references per encoder set are in the FROZEN AUDIT table of Cell 3.)")

def paired(split, k, a_enc, a_head, b_enc, b_head):
    g = qdf[(qdf["split"] == split) & (qdf["shot"] == k)]
    A = g[(g["encoder"] == a_enc) & (g["head"] == a_head)].set_index("episode")["macro_auroc"]   # g.head is DataFrame.head()
    B = g[(g["encoder"] == b_enc) & (g["head"] == b_head)].set_index("episode")["macro_auroc"]
    if len(A) == 0 or len(B) == 0: raise RuntimeError(f"empty comparison {a_enc}/{a_head} vs {b_enc}/{b_head}")
    d = (A - B).dropna().to_numpy()
    return {"delta": d.mean(), "ci95": ci95(d), "p_perm": signflip_p(d), "wins": int((d > 0).sum()), "n": len(d)}

tests = []
for s, k in product(Q_TEST_SPLITS, Q_SHOTS):
    for enc in Q_ENCODERS:
        for fam, pairs_ in (("primary", PRIMARY_TESTS), ("secondary", SECONDARY_TESTS), ("vs_prior", PRIOR_TESTS)):
            for a, b in pairs_:
                tests.append({"family": fam, "split": s, "shot": k, "comparison": f"{enc}: {a} vs {b}", **paired(s, k, enc, a, enc, b)})
    for h in HEADS:
        if {"original", "preadapted"} <= set(Q_ENCODERS):
            tests.append({"family": "preadaptation", "split": s, "shot": k, "comparison": f"{h}: preadapted vs original", **paired(s, k, "preadapted", h, "original", h)})
tdf = pd.DataFrame(tests)
tdf["p_holm"] = np.nan
for (fam, s), g in tdf.groupby(["family", "split"]):
    tdf.loc[g.index, "p_holm"] = holm(g.p_perm.to_numpy())
tdf.to_csv(QDIR / "significance.csv", index=False)
print("\n=== PAIRED TESTS (episode-level, exact/sign-flip permutation, Holm within family x split) ===\n"
      + tdf.round(4).to_string(index=False))
gates = pd.DataFrame([r for r in Q_LOG if r["head"] == "H2_hybrid" and "gate" in r])
if len(gates):
    print("\nHybrid gate towards quantum (mean over episodes):",
          {f"{e}|{k}": np.round(np.mean(np.stack(g.gate.to_list()), 0), 3).tolist() for (e, k), g in gates.groupby(["encoder", "shot"])})
qlog = pd.DataFrame(Q_LOG); qrows = qlog[qlog["head"].isin(QUANTUM_HEADS)]
if len(qrows) and not ((qrows["backend"] == "pennylane") & (qrows["qnode_calls"] > 0)).all():
    raise RuntimeError("Some quantum runs lack PennyLane provenance.")
if QSTATS["torch_sim_calls"]: raise RuntimeError("torch simulator was used outside the equivalence check.")
prov = {"backend": "pennylane", "device": PL_DEVICE_NAME, "pennylane": qml.__version__, "qubits": N_QUBITS, "layers": VQC_LAYERS,
        "qnode_calls_this_session": QSTATS["qnode_calls"], "circuit_samples_this_session": QSTATS["qnode_samples"],
        "torch_sim_calls_outside_check": QSTATS["torch_sim_calls"], "quantum_runs_verified": int(len(qrows))}
(QDIR / "quantum_provenance.json").write_text(json.dumps(prov, indent=1))
print("\n=== QUANTUM PROVENANCE ===\n" + json.dumps(prov, indent=1))
print(f"\nDONE -> {QDIR} | {elapsed()} | quantum-vs-classical and pre-adaptation claims need supervisor review.")
if SMOKE:
    mins = (time.time() - RUN_T0) / 60
    init = (qdf[qdf.split == "dev"].groupby(["encoder", "head"]).macro_auroc.mean().unstack("encoder").round(3))
    print("\n" + "=" * 78 + f"\nSMOKE INITIAL RESULT ({mins:.1f} min, budget {SMOKE_BUDGET_MIN} min)"
          + ("" if mins <= SMOKE_BUDGET_MIN else "  <-- over budget") + "\n" + "=" * 78)
    print("Frozen audit (DEV):\n" + AUDIT[AUDIT.split == "dev"].round(3).to_string(index=False))
    print(f"\n{Q_SHOTS[0]}-shot DEV macro AUROC by head (mean of {Q_EPISODES} episodes):\n" + init.to_string())
    print("\nThese numbers come from ~40-patient cohorts and a 4-qubit/1-layer circuit trained 2 epochs:"
          "\nthey only show the pipeline runs end to end. Set RUN_MODE = 'FULL' in Cell 1 and Run All.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 70.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 930.8/930.8 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 99.3 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/pennylane/__init__.py:196: RuntimeWarning: PennyLane is not yet compatible with JAX versions > 0.6.2. You have version 0.7.2 installed. Please downgrade JAX to 0.6.2 to avoid runtime errors using python -m pip install jax~=0.6.0 jaxlib~=0.6.0
  warnings.warn(


PennyLane 0.42.3 | device default.qubit (statevector simulator) | interface torch, diff_method backprop
Parameters: {'H0_classical_matchQ': '7,585', 'H0_classical_matchHybrid': '15,845', 'H0_classical_wide': '37,685', 'H1_quantum': '7,955', 'H2_hybrid': '15,545'} | qubits=6 layers=3 backend=pennylane
  H0_classical_matchQ / H1_quantum parameter ratio = 0.953
  H0_classical_matchHybrid / H2_hybrid parameter ratio = 1.019
[equivalence] torch reference vs PennyLane default.qubit: max |Δlogit| = 2.09e-07
[circuit] class 0 VQC as executed by PennyLane:
0: ──RY(0.18)───RZ(0.00)───RY(-0.00)──RZ(0.08)──╭●─────────────╭X──RY(0.18)───RZ(0.00)─ ···
1: ──RY(-0.17)──RZ(0.53)───RY(-0.02)──RZ(-0.03)─╰X─╭●──────────│───RY(-0.17)──RZ(0.53)─ ···
2: ──RY(-0.15)──RZ(0.24)───RY(0.03)───RZ(-0.02)────╰X─╭●───────│───RY(-0.15)──RZ(0.24)─ ···
3: ──RY(-0.36)──RZ(-0.20)──RY(0.07)───RZ(0.01)────────╰X─╭●────│───RY(-0.36)──RZ(-0.20) ···
4: ──RY(-0.06)──RZ(0.14)───RY(-0.03)──RZ(0.00)───────────╰X─╭●─│───RY(-0.06)──